# Curso IA Commercial — Cuaderno 03: Compendio Técnico de IA Generativa

Este cuaderno interactivo complementa el **Cuaderno 03** del curso. En este entorno reproducible puedes experimentar directamente con las matemáticas, la optimización, el cálculo de KV Cache y la construcción de un pipeline RAG vectorial.

**Temas cubiertos:**
1. Fundamentos Matemáticos: Softmax con Temperatura y Saturación de Gradiente
2. Dimensionamiento de Memoria: VRAM de Parámetros y KV Cache
3. Pipeline de RAG Vectorial con Similitud Coseno

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("¡Entorno listo para experimentar con IA Generativa!")

## 1. Función Softmax con Temperatura ($T$)

La función Softmax convierte un vector de logits $z \in \mathbb{R}^V$ en una distribución de probabilidad:

$$P(y_i) = \frac{e^{z_i / T}}{\sum_{j=1}^V e^{z_j / T}}$$

Donde:
- Si $T \to 0$, la distribución colapsa hacia la función $\text{argmax}$ (determinista, *Greedy Search*).
- Si $T = 1.0$, se preserva la distribución calibrada original del modelo.
- Si $T > 1.0$, la distribución se aplana hacia una distribución uniforme (mayor aleatoriedad y diversidad).

In [ ]:
def softmax(logits, temperature=1.0):
    z = np.array(logits) / max(temperature, 1e-5)
    # Truco numérico para evitar overflow restando el valor máximo
    exp_z = np.exp(z - np.max(z))
    return exp_z / np.sum(exp_z)

# Simulación con 5 tokens candidatos
tokens = ["Transformer", "Atención", "Red Neuronal", "Cálculo", "Estocástico"]
logits = [8.5, 7.2, 5.0, 3.1, 1.2]

plt.figure(figsize=(10, 5))
for t in [0.2, 0.7, 1.0, 2.0]:
    probs = softmax(logits, temperature=t)
    plt.plot(tokens, probs, marker='o', label=f'Temperatura T = {t}')

plt.title("Efecto de la Temperatura sobre la Distribución de Probabilidad del Vocabulario")
plt.ylabel("Probabilidad P(token)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 2. Cálculo Exacto de VRAM & KV Cache

Durante la inferencia autorregresiva de un LLM, la memoria GPU requerida se compone principalmente de:
1. **Memoria de Pesos:** $\text{VRAM}_{\text{pesos}} = \text{Parámetros (B)} \times \text{Bytes por Parámetro}$
2. **Memoria de KV Cache:** $2 \times L \times n_{\text{heads}} \times d_k \times S \times b \times \text{Bytes}$
   - Donde $L$ es el número de capas, $S$ la longitud de secuencia (contexto) y $b$ el batch size.

In [ ]:
def calcular_vram(params_billions, precision_bits=16, layers=32, heads=32, head_dim=128, seq_len=4096, batch_size=1):
    bytes_per_param = precision_bits / 8.0
    weight_vram_gb = (params_billions * 1e9 * bytes_per_param) / (1024**3)
    
    # KV Cache en GiB (2 tensores: Key y Value)
    kv_bytes = 2 * layers * heads * head_dim * seq_len * batch_size * (16 / 8.0) # KV usualmente en FP16
    kv_vram_gb = kv_bytes / (1024**3)
    
    total_vram_gb = weight_vram_gb + kv_vram_gb
    
    print(f"--- DIMENSIONAMIENTO VRAM ({params_billions}B Modelo en {precision_bits}-bit) ---")
    print(f"Memoria de Pesos del Modelo:  {weight_vram_gb:.2f} GB")
    print(f"Memoria de KV Cache (S={seq_len}): {kv_vram_gb:.2f} GB")
    print(f"VRAM Mínima Recomendada:      {total_vram_gb * 1.2:.2f} GB (incluye overhead CUDA del 20%)")
    return total_vram_gb

# Prueba para un modelo de 8B (ej. LLaMA 3 8B) en FP16 e INT4
print("\n=== Inferencia FP16 (16-bit) ===")
calcular_vram(8, precision_bits=16, seq_len=8192)

print("\n=== Inferencia Cuantizada INT4 (4-bit) ===")
calcular_vram(8, precision_bits=4, seq_len=8192)

## 3. Mini Pipeline de RAG Vectorial con Similitud Coseno

Demostración de recuperación semántica densa utilizando similitud coseno normalizada:

$$\text{Similitud}(u, v) = \cos(\theta) = \frac{u \cdot v}{\|u\| \|v\|}$$

In [ ]:
# Base de conocimiento simulada con embeddings sintéticos en d=4
documentos = [
    {"id": 1, "texto": "El mecanismo de atención permite relacionar todos los tokens en paralelo.", "vec": np.array([0.9, 0.2, 0.1, 0.05])},
    {"id": 2, "texto": "La cuantización INT4 reduce la VRAM necesaria para inferencia a una cuarta parte.", "vec": np.array([0.1, 0.85, 0.3, 0.1])},
    {"id": 3, "texto": "RAG combina recuperación de base de datos vectorial con generación condicionada.", "vec": np.array([0.4, 0.2, 0.88, 0.2])},
    {"id": 4, "texto": "Los modelos autorregresivos predicen el siguiente token de forma secuencial.", "vec": np.array([0.8, 0.3, 0.15, 0.1])}
]

def buscar_rag(query_vec, k=2):
    norm_q = np.linalg.norm(query_vec)
    resultados = []
    for doc in documentos:
        norm_d = np.linalg.norm(doc["vec"])
        sim = np.dot(query_vec, doc["vec"]) / (norm_q * norm_d)
        resultados.append((sim, doc))
    
    # Ordenar por mayor similitud
    resultados.sort(key=lambda x: x[0], reverse=True)
    return resultados[:k]

# Consulta del usuario: "¿Cómo funciona la memoria y compresión de modelos?"
query_embedding = np.array([0.15, 0.9, 0.25, 0.05])
top_k = buscar_rag(query_embedding, k=2)

print("=== RESULTADOS RECUPERADOS POR EL PIPELINE RAG ===")
for score, doc in top_k:
    print(f"Score: {score:.4f} -> [Doc {doc['id']}]: {doc['texto']}")